In [ ]:
import math

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"
import sympy as sym

from rotation.symbolic import PROP, BARE, ANGLE
from rotation.solver import initialize_pseudo, create_func_jac, solve_mixing

In [ ]:
# Mass info from online PDG (in GeV)
# https://pdglive.lbl.gov/Particle.action?node=S009
# https://pdglive.lbl.gov/Particle.action?node=S014
PDG = {
    PROP["pi0"]: 0.1349768,
    PROP["eta"]: 0.547862,
}

In [ ]:
def format_output(key, value):
    """Helps format solutions for printing"""
    if key in [PROP["pi0"], BARE["pi0"]]:
        value *= sym.Symbol(r" m_{\pi^0\text{, PDG}}") / PDG[PROP["pi0"]]
    elif key in [PROP["eta"], BARE["eta"]]:
        value *= sym.Symbol(r" m_{\eta\text{, PDG}}") / PDG[PROP["eta"]]
    elif key in [PROP["phi"], BARE["phi"], BARE["pi0-phi"], BARE["eta-phi"]]:
        value *= sym.Symbol(r" \text{ GeV}")
    elif key in ANGLE.values():
        value *= sym.Symbol(r" \pi") / math.pi
    return value

In [ ]:
argument_order, system_of_equations, partials, loaded_equation = initialize_pseudo()

In [ ]:
# Fixed known parameters 
known = {
    PROP["pi0"]: PDG[PROP["pi0"]],
    PROP["eta"]: PDG[PROP["eta"]],
    BARE["phi"]: 1.0,
    BARE["pi0-phi"]: 0.1,
    BARE["eta-phi"]: 0.3,
}

# These will be the unknowns we numerically solve for (the values are
# just initial guesses to get the solver going)
# Remember: the system of equations can only solve for up to 6 unknowns!
unknown = {
    PROP["phi"]: 1.0,
    BARE["pi0"]: PDG[PROP["pi0"]],
    BARE["eta"]: PDG[PROP["eta"]],
    ANGLE["alpha"]: 0.0,
    ANGLE["beta"]: 0.0,
    ANGLE["gamma"]: 0.0,
}

In [ ]:
# Create function and jacobian
func_jac = create_func_jac(
    system_of_equations,
    argument_order,
    list(unknown.keys()),
    known,
    partials,
)
try:
    # Solve system
    solved = solve_mixing(
        func_jac,
        argument_order,
        unknown,
    )

    # Printout the system with our known values, and unknowns blank
    print("Arguments input:")
    for k in argument_order:
        if k in known:
            sym.Eq(k, known[k], evaluate=False)
        else:
            sym.Eq(k, sym.Symbol("?"), evaluate=False)
    loaded_equation.subs(known)

    # Printout the system with our solved unknowns, and knowns blank
    print("Arguments solved:")
    for k, v in solved.items():
        v = format_output(k, v)
        sym.Eq(k, v, evaluate=False)

    print("The solved equation:")
    loaded_equation.subs(solved)

    print("Compare LHS and RHS evaluations:")
    sym.Eq(
        (1*loaded_equation.lhs.subs(known|solved)).evalf(),
        (1*loaded_equation.rhs.subs(known|solved)).evalf(),
        evaluate=False
    )

except ValueError as e:
    print(e)
    print("The solver failed! Try other inputs.")